# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, referencing every field, record set, and column using their `@id` attributes.

### Dataset Source
The dataset source Croissant schema is provided by the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant


## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an object with .name, .description, .version etc. attributes

print(f"Loaded '{metadata.name}'\nDescription: {metadata.description}\nVersion: {metadata.version}")

## 2. Data Overview

Review available record sets and their fields, with all entities referenced by their `@id` fields. Here we list the `@id` of the available record sets, with their columns and fields.


In [ ]:
# Retrieve all record sets (@id) available in the dataset
record_sets = list(dataset.metadata.record_sets)
if not record_sets:
    raise Exception("No record sets found in the dataset metadata.")
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id}")
    print(f"    name: {rs.name}")
    print(f"    description: {rs.description}")
    # List fields for this record set
    print(f"    Fields and Columns by @id:")
    if hasattr(rs, 'fields') and rs.fields:
        for field in rs.fields:
            print(f"      Field @id: {field.id} (name: {field.name}, type: {field.data_type})")
            if hasattr(field, 'column') and field.column:
                if isinstance(field.column, list):
                    for col in field.column:
                        print(f"        Column @id: {col.id} (name: {col.name})")
                else:
                    print(f"        Column @id: {field.column.id} (name: {field.column.name})")
    else:
        print("     (No fields found)")
    print()

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis, using `@id` fields only. You can adapt this to any record set as desired.

> Note: The main protein results are typically in the largest table-like record set (e.g., 'cr:main_table' if present). Identify your target record set from the previous cell.

In [ ]:
# To extract: collect all record set @id's
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

print('Extracting available record sets:')
for record_set_id in record_set_ids:
    print(f"- Extracting record set {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show one DataFrame's columns & preview (choose the largest/primary)
main_record_set_id = record_set_ids[0]  # Adjust if there is a preferred record set
print(f"\nColumns for {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using fields referenced by their `@id`. This step illustrates numerical filtering, normalization, and grouping. You can repeat this by editing field IDs, thresholds or groupings as relevant to the dataset.


In [ ]:
import numpy as np

# Example: Choose a numeric field by its @id
# Find a numeric (float or int) field
num_field_id = None
group_field_id = None
for field in dataset.metadata.record_sets[0].fields:
    if hasattr(field, 'data_type') and field.data_type in ('schema:Float', 'schema:Integer') and not num_field_id:
        num_field_id = field.id
    if not group_field_id and hasattr(field, 'data_type') and 'description' in field.name.lower():
        group_field_id = field.id
# Fallback if none matched
if not num_field_id:
    num_field_id = dataframes[main_record_set_id].select_dtypes("number").columns[0]
# By default group on accession/description/any string field
if not group_field_id:
    for field in dataset.metadata.record_sets[0].fields:
        if hasattr(field, 'data_type') and field.data_type == 'schema:Text':
            group_field_id = field.id
            break

print(f"Using numeric field @id for analysis: {num_field_id}")
print(f"Grouping by field @id: {group_field_id}")

df = dataframes[main_record_set_id]
# Handle missing/non-numeric
df[num_field_id] = pd.to_numeric(df[num_field_id], errors='coerce')

threshold = df[num_field_id].dropna().quantile(0.75)  # 75th percentile as threshold example
filtered_df = df[df[num_field_id] > threshold].copy()
print(f"Filtered records with {num_field_id} > {threshold:.3f} (top quartile):\n")
print(filtered_df[[num_field_id]].head())

filtered_df[f"{num_field_id}_normalized"] = (filtered_df[num_field_id] - filtered_df[num_field_id].mean()) / filtered_df[num_field_id].std()
print(f"\nNormalized {num_field_id} for filtered records:")
print(filtered_df[[num_field_id, f"{num_field_id}_normalized"].head()])

# Group by group_field_id if present
if group_field_id and group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[num_field_id].mean().reset_index()
    print(f"\nGrouped means by {group_field_id}:")
    print(grouped_df.head())


## 5. Visualization
Visualize data distributions using field `@id` values only. Here we show a histogram and a boxplot of the normalized numeric field, and a barplot for the group mean if grouping is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(filtered_df) > 0:
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(filtered_df[num_field_id], bins=30, ax=axs[0], kde=True)
    axs[0].set_title(f"Histogram: {num_field_id}")
    sns.boxplot(x=filtered_df[num_field_id], ax=axs[1])
    axs[1].set_title(f"Boxplot: {num_field_id}")
    plt.tight_layout()
    plt.show()
    # Group plot if available
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10,4))
        grouped_df = filtered_df.groupby(group_field_id)[num_field_id].mean().sort_values(ascending=False).head(15)
        sns.barplot(y=grouped_df.index, x=grouped_df.values)
        plt.title(f"Mean {num_field_id} by {group_field_id}")
        plt.xlabel(num_field_id)
        plt.ylabel(group_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR\textsuperscript{2} mass spectrometry protein dataset using the `mlcroissant` library. We demonstrated how to reference all data structures by their Croissant `@id`, identified main numeric fields, performed basic filtering and normalization, and visualized key patterns within the dataset. You can use these code blocks to further analyze, process, and visualize additional fields of interest from this dataset or other Croissant-compliant resources.
